# 05 — Clustering de Países por Perfil de Inclusión Digital
**Fase CRISP-DM:** Modelado  
**Responde a:** P4 — ¿Existen tipologías diferenciadas de modelos de inclusión digital?  
**Técnica:** K-Means (scikit-learn)

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import sys
sys.path.insert(0, '..')
from utils import setup_style, PALETTE, COUNTRY_NAMES_ES, save_figure
setup_style()

df_m = pd.read_csv('../data/processed/dsb_derived_metrics.csv')
print('Datos cargados:', df_m.shape)

## 1. Preparación de la matriz de features

In [ ]:
# Excluir EU27_2020 (es un agregado, no un país)
df_c = df_m[df_m.pais_cod != 'EU27_2020'].copy()

# Features para el clustering
features = ['gap_total','gap_genero','gap_edad','f_dis_sev','y55_74_dis_sev']
X = df_c[features].fillna(df_c[features].median())
X_scaled = StandardScaler().fit_transform(X)

print('Países incluidos:', df_c['pais_cod'].tolist())
print('Features:', features)
print('Shape de X:', X_scaled.shape)

## 2. Selección de K — método del codo

In [ ]:
inertias = []
scores = []
K_range = range(2, min(len(df_c), 6))
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    scores.append(silhouette_score(X_scaled, km.labels_))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(K_range, inertias, 'o-', color=PALETTE['blue'])
ax1.set_title('Método del Codo', fontweight='bold')
ax1.set_xlabel('K'); ax1.set_ylabel('Inercia')
ax2.plot(K_range, scores, 's-', color=PALETTE['green'])
ax2.set_title('Silhouette Score', fontweight='bold')
ax2.set_xlabel('K'); ax2.set_ylabel('Score')
plt.tight_layout()
save_figure(fig, 'fig_clustering_elbow.png')
plt.show()

## 3. K-Means con K=3

In [ ]:
# Ajustar K según el análisis del codo
K_OPTIMO = 3
km = KMeans(n_clusters=K_OPTIMO, random_state=42, n_init=10)
df_c['cluster'] = km.fit_predict(X_scaled)

# Etiquetas interpretativas
label_map = {}  # Completar tras ver los resultados
print(df_c[['pais_cod','pais_nombre_es','gap_total','cluster']].sort_values('cluster').to_string(index=False))

## 4. Visualización del clustering

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = [PALETTE['green'], PALETTE['amber'], PALETTE['red']]
for cluster_id in sorted(df_c['cluster'].unique()):
    mask = df_c['cluster'] == cluster_id
    ax.scatter(df_c.loc[mask,'gap_total'], df_c.loc[mask,'f_dis_sev'],
               c=colors[cluster_id], s=120, zorder=3, label=f'Grupo {cluster_id+1}')
    for _, row in df_c[mask].iterrows():
        ax.annotate(row['pais_cod'], (row['gap_total'], row['f_dis_sev']),
                    xytext=(4, 4), textcoords='offset points', fontsize=9)
ax.set_xlabel('Brecha total (pp)', fontsize=10)
ax.set_ylabel('% mujeres con discapacidad severa', fontsize=10)
ax.set_title('Clustering de países europeos por perfil de inclusión digital\n(K-Means, K=3)', fontweight='bold', color=PALETTE['navy'])
ax.legend()
fig.text(0.01, -0.02, 'Fuente: elaboración propia a partir de Eurostat DSB_ICTIU01 (2024).', fontsize=8)
plt.tight_layout()
save_figure(fig, 'fig_clustering_scatter.png')
plt.show()

## 5. Interpretación de los grupos

> **Completar tras ver los resultados del clustering:**

- **Grupo A (alta inclusión):** ...
- **Grupo B (inclusión media):** ...
- **Grupo C (baja inclusión):** ...